# **FINETUNING**

## Environment, paths and input availability

In [ ]:

# ============================================================
# BLOCK 1 — ENVIRONMENT AND INPUT AVAILABILITY
# CTDDG / CDGCN Protein-Conditioned Fine-Tuning
# ============================================================

import os
import sys
import platform
import hashlib
import importlib.util
from importlib.metadata import version, PackageNotFoundError
from pathlib import Path

# ------------------------------------------------------------
# 1. Experiment configuration
# ------------------------------------------------------------

PRETRAINED_VARIANT = "no_toxicity"
BINDINGDB_FOLD = "1036_124"

EXPECTED_N_ATOM = 55
EXPECTED_N_BOND = 4
EXPECTED_PROTEIN_DIM = 6165

EXPECTED_VOCAB_SHA256 = (
    "3f2a10321e1301f7a45f6ebb260181c70c0e28ae5"
    "c3162bd5c3e713db335184f"
)

RUN_TRAINING = False
RUN_TEST_EVALUATION = False

In [ ]:
# ------------------------------------------------------------
# 2. Project directories
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    os.environ.get(
        "CTDDG_REPO_ROOT",
        "/home/muhamed/repo/CTDDG"
    )
).expanduser().resolve()

REWORKED_ROOT = PROJECT_ROOT / "REWORKED_CODE"

PREPROCESSING_ROOT = REWORKED_ROOT / "PREPROCESSING_OUTPUT"
PRETRAINING_ROOT = REWORKED_ROOT / "PRETRAINING_OUTPUT"
FINETUNING_ROOT = REWORKED_ROOT / "finetuning"
EVALUATION_ROOT = REWORKED_ROOT / "EVALUATION_OUTPUT"



In [ ]:
# ------------------------------------------------------------
# 3. Pretrained checkpoints
# ------------------------------------------------------------

PRETRAINED_CHECKPOINTS = {
    "no_toxicity": (
        PRETRAINING_ROOT
        / "no_toxicity/training_runs"
        / "chembl_B32_K5_alpha08_seed42/best.pt"
    ),
    "toxicity": (
        PRETRAINING_ROOT
        / "toxicity_aware/runs"
        / "ctddg_main_bs32_lambda2/model_best.pt"
    ),
}

assert PRETRAINED_VARIANT in PRETRAINED_CHECKPOINTS

CHECKPOINT_PATH = PRETRAINED_CHECKPOINTS[PRETRAINED_VARIANT]
PRETRAIN_RUN_DIR = CHECKPOINT_PATH.parent



In [ ]:
# ------------------------------------------------------------
# 4. BindingDB datasets and vocabulary
# ------------------------------------------------------------

ATOM_TYPES_PATH = (
    PREPROCESSING_ROOT / "vocabulary/atom_types.txt"
)

TRAIN_DATA_PATH = (
    PREPROCESSING_ROOT
    / "bindingdb/train"
    / f"train_4_org_{BINDINGDB_FOLD}"
    / "pairs_clean.csv.gz"
)

TEST_DATA_PATH = (
    PREPROCESSING_ROOT
    / "bindingdb/test"
    / f"test_4_org_{BINDINGDB_FOLD}"
    / "pairs_clean.csv.gz"
)



In [ ]:
# ------------------------------------------------------------
# 5. ProSE protein embedding locations
# ------------------------------------------------------------

TRAIN_EMBEDDING_DIR = (
    EVALUATION_ROOT
    / "embeddings/train"
    / BINDINGDB_FOLD
)

TRAIN_EMBEDDING_H5 = (
    TRAIN_EMBEDDING_DIR
    / "prose_embeddings_avg_6165.h5"
)

TRAIN_PROTEIN_MANIFEST = (
    TRAIN_EMBEDDING_DIR / "protein_manifest.csv"
)

TRAIN_PAIR_MAPPING = (
    TRAIN_EMBEDDING_DIR / "pair_to_protein_index.csv"
)

# Test embedding location must be independently verified.
TEST_EMBEDDING_CANDIDATES = [
    EVALUATION_ROOT / "embeddings" / BINDINGDB_FOLD,
    EVALUATION_ROOT / "embeddings/test" / BINDINGDB_FOLD,
]



In [ ]:
# ------------------------------------------------------------
# 6. Environment inspection
# ------------------------------------------------------------

print("=" * 75)
print("BLOCK 1 — ENVIRONMENT")
print("=" * 75)

print("Python version :", sys.version.split()[0])
print("Interpreter    :", sys.executable)
print("Hostname       :", platform.node())
print("SLURM job ID   :", os.environ.get("SLURM_JOB_ID", "None"))
print("CUDA devices   :", os.environ.get("CUDA_VISIBLE_DEVICES", "Not set"))

required_modules = {
    "torch": "torch",
    "numpy": "numpy",
    "pandas": "pandas",
    "rdkit": "rdkit",
    "scipy": "scipy",
    "networkx": "networkx",
    "h5py": "h5py",
    "scikit-learn": "sklearn",
}

missing_packages = []

print("\nPackage availability:")

for package, module in required_modules.items():
    found = importlib.util.find_spec(module) is not None

    if found:
        try:
            package_version = version(package)
        except PackageNotFoundError:
            package_version = "version unavailable"

        print(f"  OK      {package:15s} {package_version}")
    else:
        print(f"  MISSING {package}")
        missing_packages.append(package)

if "torch" not in missing_packages:
    import torch

    print("\nPyTorch CUDA build :", torch.version.cuda)
    print("CUDA available     :", torch.cuda.is_available())

    if torch.cuda.is_available():
        print("GPU                :", torch.cuda.get_device_name(0))
        print(
            "GPU memory         :",
            round(
                torch.cuda.get_device_properties(0).total_memory
                / 1024**3, 2
            ),
            "GiB"
        )

# ------------------------------------------------------------
# 7. Read-only file inventory
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("BLOCK 1 — REQUIRED INPUT FILES")
print("=" * 75)

required_files = {
    "Project root": PROJECT_ROOT,
    "Reworked root": REWORKED_ROOT,
    "Pretrained checkpoint": CHECKPOINT_PATH,
    "Atom vocabulary": ATOM_TYPES_PATH,
    "BindingDB training pairs": TRAIN_DATA_PATH,
    "BindingDB held-out test pairs": TEST_DATA_PATH,
}

missing_required = []

for name, path in required_files.items():
    exists = path.exists()

    print(f"\n{'OK' if exists else 'MISSING'} — {name}")
    print(f"  {path}")

    if not exists:
        missing_required.append(name)
    elif path.is_file():
        print(f"  Size: {path.stat().st_size / 1024**2:.2f} MiB")

print("\n" + "=" * 75)
print("BLOCK 1 — PROTEIN EMBEDDINGS")
print("=" * 75)

embedding_files = {
    "Training HDF5": TRAIN_EMBEDDING_H5,
    "Protein manifest": TRAIN_PROTEIN_MANIFEST,
    "Pair-to-protein mapping": TRAIN_PAIR_MAPPING,
    "ProSE input FASTA": TRAIN_EMBEDDING_DIR / "prose_input.fasta",
    "Embedding metadata": TRAIN_EMBEDDING_DIR / "prose_run_metadata.json",
}

missing_embeddings = []

for name, path in embedding_files.items():
    exists = path.is_file()
    print(f"{'OK' if exists else 'MISSING':8s} {name}: {path}")

    if not exists:
        missing_embeddings.append(name)

print("\nPossible test embedding directories:")

for path in TEST_EMBEDDING_CANDIDATES:
    print(f"  {'FOUND' if path.is_dir() else 'NOT FOUND'}: {path}")

In [ ]:
# ------------------------------------------------------------
# 8. Verify ordered atom vocabulary
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("BLOCK 1 — VOCABULARY IDENTITY")
print("=" * 75)

vocab_verified = False

if ATOM_TYPES_PATH.is_file():

    raw_vocab = ATOM_TYPES_PATH.read_bytes()

    vocab_sha256 = hashlib.sha256(raw_vocab).hexdigest()

    atom_types = [
        line.strip()
        for line in raw_vocab.decode("utf-8-sig").splitlines()
        if line.strip()
    ]

    print("Atom types     :", len(atom_types))
    print("Vocabulary SHA :", vocab_sha256)
    print("Expected SHA   :", EXPECTED_VOCAB_SHA256)

    vocab_verified = (
        len(atom_types) == EXPECTED_N_ATOM
        and vocab_sha256 == EXPECTED_VOCAB_SHA256
        and len(set(atom_types)) == len(atom_types)
    )

    print("Vocabulary verified:", vocab_verified)

else:
    print("Vocabulary not available.")

# ------------------------------------------------------------
# 9. Final preflight summary
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("BLOCK 1 — SUMMARY")
print("=" * 75)

print("Selected checkpoint :", PRETRAINED_VARIANT)
print("BindingDB fold      :", BINDINGDB_FOLD)
print("Protein dimensions  :", EXPECTED_PROTEIN_DIM)
print("Required files missing:", len(missing_required))
print("Embedding files missing:", len(missing_embeddings))
print("Missing packages    :", missing_packages)
print("Vocabulary verified :", vocab_verified)
print("Training enabled    :", RUN_TRAINING)
print("Test evaluation     :", RUN_TEST_EVALUATION)
print("Files modified      : NONE")

if (
    not missing_required
    and not missing_packages
    and vocab_verified
):
    print("\nSTATUS: Ready for detailed checkpoint and data inspection.")
else:
    print("\nSTATUS: HOLD — Review missing or mismatched inputs.")

if missing_embeddings:
    print(
        "NOTE: Training embeddings are incomplete or not at "
        "the expected locations. Verify before fine-tuning."
    )

print("=" * 75)
